### Model adjustment process

In [1]:
import os
from transformers import BeitForImageClassification, BeitFeatureExtractor
from PIL import Image
import torch
import torch.nn as nn
from torchinfo import summary


In [11]:
# Load original model (still has 21841 classes)
raw_model_path = "/Volumes/KODAK/folder_02/Skin_cancer_Detection/model/Raw_model/beit_base_patch16_model"
model = BeitForImageClassification.from_pretrained(raw_model_path)

# Assign correct label mappings
class_labels = ['AKIEC', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'VASC']
model.config.id2label = {i: label for i, label in enumerate(class_labels)}
model.config.label2id = {label: i for i, label in enumerate(class_labels)}
model.config.num_labels = len(class_labels)  #  Ensure num_labels is set

# Resize classifier layer (IMPORTANT STEP)
model.classifier = nn.Linear(model.classifier.in_features, model.config.num_labels)

# Set to eval mode
model.eval()

# Save adjusted model
adjusted_model_path = "/Volumes/KODAK/folder_02/Skin_cancer_Detection/model/pre-prcess model/adjusted_beit_model"
os.makedirs(adjusted_model_path, exist_ok=True)
model.save_pretrained(adjusted_model_path)

# Save feature extractor
feature_extractor = BeitFeatureExtractor.from_pretrained(raw_model_path)
feature_extractor.save_pretrained(adjusted_model_path)

print(" Model successfully adjusted to 7 classes and saved.")


 Model successfully adjusted to 7 classes and saved.


In [13]:
# Ensure adjusted model have seven class


adjusted_model_path = "/Volumes/KODAK/folder_02/Skin_cancer_Detection/model/pre-prcess model/adjusted_beit_model"
model = BeitForImageClassification.from_pretrained(adjusted_model_path)

print(" Number of output classes:", model.config.num_labels)
print(" id2label mapping:", model.config.id2label)


 Number of output classes: 7
 id2label mapping: {0: 'AKIEC', 1: 'BCC', 2: 'BKL', 3: 'DF', 4: 'MEL', 5: 'NV', 6: 'VASC'}


In [21]:
# === Load adjusted model ===
model_path = "/Volumes/KODAK/folder_02/Skin_cancer_Detection/model/pre-prcess model/adjusted_beit_model"
model = BeitForImageClassification.from_pretrained(model_path)
feature_extractor = BeitFeatureExtractor.from_pretrained(model_path)

model.eval()

# === Load test image ===
test_image_path = "/Volumes/KODAK/folder_02/Skin_cancer_Detection/data/pre-process data/final_preprocess_images/nv/nv.071.jpg"  # Replace this path
image = Image.open(test_image_path).convert("RGB")

# === Preprocess image ===
inputs = feature_extractor(images=image, return_tensors="pt")

# === Inference ===
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    predicted_class_idx = logits.argmax(-1).item()

# === Print result ===
predicted_label = model.config.id2label[predicted_class_idx]
print(f"🧠 Predicted class: {predicted_label}")

🧠 Predicted class: MEL


In [ ]:
# Showing model summary

# Load your adjusted model
model_path = "/Volumes/KODAK/folder_02/Skin_cancer_Detection/model/pre-prcess model/adjusted_beit_model"
model = BeitForImageClassification.from_pretrained(model_path)

# Move model to device (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# BEiT expects input shape: (batch_size, 3, 224, 224)
input_size = (1, 3, 224, 224)  # batch_size=1 for summary

# Print model summary
summary(model, input_size=input_size, device=str(device))

Layer (type:depth-idx)                                                 Output Shape              Param #
BeitForImageClassification                                             [1, 7]                    --
├─BeitModel: 1-1                                                       [1, 768]                  --
│    └─BeitEmbeddings: 2-1                                             [1, 197, 768]             768
│    │    └─BeitPatchEmbeddings: 3-1                                   [1, 196, 768]             590,592
│    │    └─Dropout: 3-2                                               [1, 197, 768]             --
│    └─BeitEncoder: 2-2                                                [1, 197, 768]             --
│    │    └─ModuleList: 3-3                                            --                        85,169,088
│    └─Identity: 2-3                                                   [1, 197, 768]             --
│    └─BeitPooler: 2-4                                                 [1, 768]  

In [ ]:
# Model unfreezing process


# === Paths ===
adjusted_model_path = "/Volumes/KODAK/folder_02/Skin_cancer_Detection/model/pre-prcess model/adjusted_beit_model"
unfrozen_model_path = "/Volumes/KODAK/folder_02/Skin_cancer_Detection/model/pre-prcess model/unfrozen_beit_model"
os.makedirs(unfrozen_model_path, exist_ok=True)

# === Load the model ===
model = BeitForImageClassification.from_pretrained(adjusted_model_path)

# === Unfreeze all layers ===
for name, param in model.named_parameters():
    param.requires_grad = True
print("All layers have been set to trainable (requires_grad=True).")

# === Print parameter stats ===
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# === Print number of output classes ===
print(f"Number of output classes: {model.config.num_labels}")
print(f"Class label mapping (id2label): {model.config.id2label}")

# === Save unfrozen model ===
model.save_pretrained(unfrozen_model_path)

# === Save the corresponding feature extractor ===
feature_extractor = BeitFeatureExtractor.from_pretrained(adjusted_model_path)
feature_extractor.save_pretrained(unfrozen_model_path)

print(f"Unfrozen model and feature extractor saved to:\n{unfrozen_model_path}")


✅ All layers have been set to trainable (requires_grad=True).
🔢 Total parameters:     85,767,367
🧠 Trainable parameters: 85,767,367
🧾 Number of output classes: 7
📌 Class label mapping (id2label): {0: 'AKIEC', 1: 'BCC', 2: 'BKL', 3: 'DF', 4: 'MEL', 5: 'NV', 6: 'VASC'}
✅ Unfrozen model and feature extractor saved to:
/Volumes/KODAK/folder_02/Skin_cancer_Detection/model/pre-prcess model/unfrozen_beit_model
